# 30 · Vector & graph — Qdrant dedicated vector search

**Qdrant is the mesh's purpose-built vector database.** Where the query wave (notebooks
`20`/`21`/`22`) answers questions in SQL over rows and columns, this notebook answers a
different *kind* of question: **"what is nearest to this, in meaning?"** — approximate
nearest-neighbour (ANN) search over high-dimensional embedding vectors, the retrieval
engine behind RAG, semantic search, and recommendation.

You can bolt vectors onto a general store — the platform also runs **pgvector** inside the
operational Postgres, vectors living beside relational rows. Qdrant is the other choice: a
store that does **only** vectors and therefore does them at scale — a filterable HNSW index,
a payload document attached to every point, on-disk and quantized variants, and a search API
built around the vector, not bolted onto SQL.

> **SQL asks "which rows match?" Qdrant asks "which vectors are nearest?"** That is the whole
> reason a dedicated vector store earns a place in the mesh next to the SQL engines.

### What this notebook does

It connects to Qdrant, **discovers** the live collections and reads each one's *real* vector
geometry (dimension, distance metric, HNSW config) rather than assuming it, then runs a real
**semantic similarity search** and a **payload-filtered** (metadata-constrained) search over
actual data.

> **Read-only, throughout.** Every call here is a `get_collections` / `get_collection` /
> `scroll` / `retrieve` / `query_points` — discovery and search only. Nothing is **upserted,
> deleted, or re-indexed**; the collections are read exactly as the hydration left them, so —
> as in notebooks `20`/`22` — there is **no cleanup section.** And we never *assume* a
> collection's dimension or payload fields: each is **discovered from the live store first**,
> then the search is built around what was found.

## Setup

The `qdrant-client` library is **not** in the singleuser base image (which ships `polars`,
`s3fs`, `pyarrow`, `duckdb`, `fastavro`), so we install it here. `polars` — used to render
result frames, exactly as in notebooks `20`/`22` — already ships in the image.

In [1]:
%pip install -q qdrant-client

Note: you may need to restart the kernel to use updated packages.


## Connect

Connection is **env-driven**, the same pattern the query notebooks use. The committed default
is the **in-cluster** service URL (`qdrant.weyland.svc.cluster.local:6333`); a validation run
overrides `QDRANT_URL` via the environment (e.g. to a NodePort) **without editing the
notebook**. So the notebook never captures the resolved address — the connection is proven by
the **collection list it returns**, not by echoing the endpoint.

This Qdrant is **unmeshed and unauthenticated** (open on the LAN), so the client needs only a
URL — no API key. It speaks HTTP/REST on `6333` (gRPC is `6334`).

In [2]:
import os
from qdrant_client import QdrantClient, models
import polars as pl

QDRANT_URL = os.environ.get("QDRANT_URL", "http://qdrant.weyland.svc.cluster.local:6333")
client = QdrantClient(url=QDRANT_URL)

# small display helpers (house style: render frames with polars, as in notebooks 20/22)
def dist_name(d):                       # Distance enum -> plain string ("Cosine")
    return getattr(d, "value", str(d))
def snippet(text, n=90):                # flatten whitespace + truncate long text payloads
    return None if text is None else " ".join(str(text).split())[:n]

# Prove the connection by WHAT IT RETURNS, never by echoing the resolved endpoint.
collections = sorted(c.name for c in client.get_collections().collections)
print(f"connected to Qdrant - {len(collections)} collections")
for name in collections:
    print("  -", name)

connected to Qdrant - 11 collections
  - datasets_health_big_five
  - datasets_health_open_food_facts
  - datasets_music_audioset
  - datasets_music_fma_echonest
  - datasets_music_fma_features
  - datasets_music_gtzan
  - datasets_music_lp_musiccaps_mc
  - datasets_music_lp_musiccaps_mtt
  - datasets_music_spotify_tracks
  - datasets_music_uci_year_prediction
  - weyland_chunks


## Discover — what collections exist, and what shape are they?

Qdrant groups vectors into **collections**, each with a fixed **dimension** and **distance
metric** chosen when it was created. `get_collections()` lists them; `get_collection(name)`
reads one collection's real configuration. This store holds **two families**:

- **`weyland_chunks`** — the RAG corpus: chunked text from the lab's own docs and code,
  embedded to `768`-dim vectors. This is the richest collection and the one we search below.
- **ten `datasets_*` collections** — per-dataset feature vectors (music audio features, health
  survey factors), each with its own small dimension.

We read every collection's dimension, distance and point count straight from the server — the
authoritative shape, never an assumption.

In [3]:
def collection_row(name):
    info = client.get_collection(name)
    v = info.config.params.vectors          # single unnamed vector per collection here
    return {"collection": name, "dim": v.size,
            "distance": dist_name(v.distance), "points": info.points_count}

overview = pl.DataFrame([collection_row(n) for n in collections]).sort("points", descending=True)
overview

collection,dim,distance,points
str,i64,str,i64
"""datasets_music_uci_year_predic…",90,"""Cosine""",515345
"""datasets_health_open_food_fact…",384,"""Cosine""",200000
"""datasets_music_spotify_tracks""",11,"""Cosine""",114000
"""datasets_music_fma_features""",518,"""Cosine""",106574
"""datasets_music_audioset""",384,"""Cosine""",35824
…,…,…,…
"""datasets_health_big_five""",50,"""Cosine""",19719
"""datasets_music_fma_echonest""",244,"""Cosine""",14511
"""weyland_chunks""",768,"""Cosine""",8309


## Inspect a collection — the real vector geometry

Before searching, read the exact geometry of the collection we'll query. Three things define
how similarity works here:

- **size `768`** — the embedding dimension; every vector is a point in 768-dimensional space,
  and any query vector must match that width.
- **distance `Cosine`** — nearness is cosine similarity (angle between vectors), so scores run
  ~`-1..1` and `1.0` is an exact direction match. (The other options are `Euclid` and `Dot`.)
- **HNSW index** — Hierarchical Navigable Small World, the graph Qdrant walks to find nearest
  neighbours *approximately* instead of scanning every point. `m` and `ef_construct` are its
  build-time knobs; `full_scan_threshold` is the point count below which Qdrant just
  brute-forces instead.

In [4]:
info = client.get_collection("weyland_chunks")
vp   = info.config.params.vectors
hnsw = info.config.hnsw_config

print("weyland_chunks")
print("  vector size        :", vp.size)
print("  distance           :", dist_name(vp.distance))
print("  points             :", info.points_count)
print("  HNSW m             :", hnsw.m)
print("  HNSW ef_construct  :", hnsw.ef_construct)
print("  full_scan_threshold:", hnsw.full_scan_threshold)
print("  on_disk            :", hnsw.on_disk)
print("  quantization       :", info.config.quantization_config)

weyland_chunks
  vector size        : 768
  distance           : Cosine
  points             : 8309
  HNSW m             : 16
  HNSW ef_construct  : 100
  full_scan_threshold: 10000
  on_disk            : False
  quantization       : None


## Payload — the metadata that travels with every vector

A Qdrant point is a **vector plus a payload**: an arbitrary JSON document of metadata stored
*alongside* the vector. The payload is what makes results usable (you get the source text back,
not just an id) and what makes **filtering** possible. We `scroll` a few points — a
non-scoring page through the collection — to see the real payload shape before we rely on any
field name.

For `weyland_chunks` the payload carries where the chunk came from (`source_name`,
`source_path`), its position (`chunk_index`, `chunk_title`), and the `content` text itself.

In [5]:
recs, _ = client.scroll("weyland_chunks", limit=3, with_payload=True, with_vectors=False)
pl.DataFrame([
    {"id": str(r.id)[:8],
     "source_name": r.payload.get("source_name"),
     "chunk_index": r.payload.get("chunk_index"),
     "chunk_title": r.payload.get("chunk_title"),
     "content": snippet(r.payload.get("content"))}
    for r in recs
])

id,source_name,chunk_index,chunk_title,content
str,str,i64,str,str
"""0002ea0b""","""arch.md""",13,"""13. Roadmap & maintenance""","""## 13. Roadmap & maintenance F…"
"""00163a46""","""music-marts-overview.yml""",1,null,""": chartName: Genre acousticnes…"
"""001b018e""","""structured-prompt-driven-devel…",6,"""Engineering Knowledge Statemen…","""## Engineering Knowledge State…"


## Semantic similarity search — nearest neighbours of a vector

This is what a vector store is *for*: given a query vector, return the points whose vectors are
nearest. In production the query vector comes from embedding a user's question with the same
model that built the collection — **but this notebook deliberately carries no embedding model**
(that would mean shipping a transformer and asserting which model matches these 768-dim
vectors). So we prove ANN over real data a cleaner way:

> **Search by an existing point's own vector.** Pull one point back *with* its stored vector
> (`with_vectors=True`), then ask Qdrant for that vector's nearest neighbours. No embedding
> step, no assumption about the model — just the index doing its job over real vectors.

The seed point comes back as its own top hit at **score `1.0`** (a vector is maximally similar
to itself — a built-in proof the search is exact about identity); the rows beneath it are the
genuine nearest neighbours.

In [6]:
# pull ONE point back with its stored vector, then search for that vector's neighbours
seed_page, _ = client.scroll("weyland_chunks", limit=1, with_payload=True, with_vectors=True)
seed = seed_page[0]
seed_vec = seed.vector
print("seed point :", str(seed.id)[:8], "|", seed.payload["source_name"], "|", seed.payload.get("chunk_title"))
print("query vector:", len(seed_vec), "dims (cosine)\n")

hits = client.query_points("weyland_chunks", query=seed_vec, limit=6, with_payload=True).points
pl.DataFrame([
    {"rank": i, "score": round(h.score, 4),
     "source_name": h.payload.get("source_name"),
     "chunk_title": h.payload.get("chunk_title"),
     "content": snippet(h.payload.get("content"), 70)}
    for i, h in enumerate(hits, 1)
])

seed point : 0002ea0b | arch.md | 13. Roadmap & maintenance
query vector: 768 dims (cosine)



rank,score,source_name,chunk_title,content
i64,f64,str,str,str
1,1.0,"""arch.md""","""13. Roadmap & maintenance""","""## 13. Roadmap & maintenance F…"
2,0.8377,"""port-iac-coverage.md""","""UI walkthrough (eyes-on)""","""## UI walkthrough (eyes-on) 1.…"
3,0.8189,"""image-signatures.yaml""",null,"""# NeoDash - opensearchproject/…"
4,0.8174,"""README.md""","""Outstanding (before any row is…","""## Outstanding (before any row…"
5,0.8052,"""backlog.md""",null,"""# Weyland Forward Roadmap — re…"
6,0.8046,"""datasets-hydration.md""","""Store roadmap (the grid's Tier…","""## Store roadmap (the grid's T…"


Read the frame top-down: **rank 1 is the seed itself at `1.0`**, confirming the index found the
exact point; every row below is a real neighbour, ranked by descending cosine score. These are
the passages a RAG retriever would hand to an LLM as context for a question shaped like the seed
chunk — semantically close text the store surfaced without a single keyword match.

## Payload-filtered ANN — metadata-constrained search

Pure nearest-neighbour search ignores metadata — it will happily return the closest vector from
*anywhere* in the collection. Real retrieval usually needs a **constraint**: only this document,
only this tenant, only fresh rows, only one category. Qdrant applies the filter **inside** the
HNSW traversal (filterable HNSW), so you get the nearest neighbours *that also satisfy the
filter* — not a post-hoc filter over an already-truncated result.

The most common RAG version is **scope retrieval to one source**. We take the same query vector
and add a `Filter` on `source_name` — the field we just saw in the payload — so only chunks from
that one document can be returned.

In [7]:
scope = seed.payload["source_name"]          # the seed came from this document
flt = models.Filter(must=[
    models.FieldCondition(key="source_name", match=models.MatchValue(value=scope))
])
in_scope = client.count("weyland_chunks", count_filter=flt, exact=True).count
print(f"scoping the SAME search to source_name = {scope!r}  ({in_scope} chunks in that source)\n")

hits = client.query_points("weyland_chunks", query=seed_vec, query_filter=flt,
                           limit=5, with_payload=True).points
pl.DataFrame([
    {"score": round(h.score, 4),
     "source_name": h.payload.get("source_name"),
     "chunk_title": h.payload.get("chunk_title"),
     "content": snippet(h.payload.get("content"), 70)}
    for h in hits
])

scoping the SAME search to source_name = 'arch.md'  (14 chunks in that source)



score,source_name,chunk_title,content
f64,str,str,str
1.0,"""arch.md""","""13. Roadmap & maintenance""","""## 13. Roadmap & maintenance F…"
0.788,"""arch.md""","""4. Hosts & roles""","""## 4. Hosts & roles | Host | W…"
0.783,"""arch.md""","""10. Cross-cutting concerns""","""## 10. Cross-cutting concerns …"
0.7651,"""arch.md""","""6. Component inventory""","""## 6. Component inventory ### …"
0.7614,"""arch.md""","""8. Model serving""","""## 8. Model serving | Path | W…"


Same mechanic, categorical field, different collection. `datasets_music_spotify_tracks` holds
**114k tracks** as **11-dimensional audio-feature vectors** (danceability, energy, tempo,
loudness, ...) with a `track_genre` payload. Filtering the vector search by genre is the
canonical **"find things acoustically similar to this — but only within genre X"** query.

Watch what the **unfiltered** result exposes about the real data: the seed track comes back
several times at score `1.0`, because this dataset catalogues the *same track under multiple
genre labels* (each labelling is its own point). The **genre filter** cuts straight through
that — constrain to `metal` and only metal tracks, ranked by audio-feature nearness, survive.

In [8]:
sp = "datasets_music_spotify_tracks"
sinfo = client.get_collection(sp)
print(sp, "->", sinfo.config.params.vectors.size, "dims,",
      dist_name(sinfo.config.params.vectors.distance) + ",", sinfo.points_count, "points")

s_page, _ = client.scroll(sp, limit=1, with_payload=True, with_vectors=True)
s = s_page[0]
print("seed track:", s.payload["track_name"], "-", s.payload["artists"], f"({s.payload['track_genre']})\n")

def track_rows(points):
    return pl.DataFrame([
        {"score": round(p.score, 4), "genre": p.payload.get("track_genre"),
         "track": p.payload.get("track_name"), "artists": p.payload.get("artists")}
        for p in points
    ])

unfiltered = client.query_points(sp, query=s.vector, limit=5, with_payload=True).points
genre = "metal"
gflt = models.Filter(must=[models.FieldCondition(key="track_genre",
                                                 match=models.MatchValue(value=genre))])
filtered = client.query_points(sp, query=s.vector, query_filter=gflt, limit=5, with_payload=True).points

print("UNFILTERED nearest neighbours (any genre):")
print(track_rows(unfiltered))
print(f"\nFILTERED to track_genre = {genre!r}:")
track_rows(filtered)

datasets_music_spotify_tracks -> 11 dims, Cosine, 114000 points
seed track: Comedy - Gen Hoshino (acoustic)

UNFILTERED nearest neighbours (any genre):
shape: (5, 4)
┌────────┬───────────────────┬─────────┬─────────────┐
│ score  ┆ genre             ┆ track   ┆ artists     │
│ ---    ┆ ---               ┆ ---     ┆ ---         │
│ f64    ┆ str               ┆ str     ┆ str         │
╞════════╪═══════════════════╪═════════╪═════════════╡
│ 1.0    ┆ acoustic          ┆ Comedy  ┆ Gen Hoshino │
│ 1.0    ┆ songwriter        ┆ Comedy  ┆ Gen Hoshino │
│ 1.0    ┆ j-pop             ┆ Comedy  ┆ Gen Hoshino │
│ 1.0    ┆ singer-songwriter ┆ Comedy  ┆ Gen Hoshino │
│ 0.9367 ┆ reggae            ┆ JAMAICA ┆ Feid;Sech   │
└────────┴───────────────────┴─────────┴─────────────┘

FILTERED to track_genre = 'metal':


score,genre,track,artists
f64,str,str,str
0.7554,"""metal""","""Do Ya Wanna Taste It""","""Wig Wam"""
0.6859,"""metal""","""Roulette""","""Red Hot Chili Peppers"""
0.658,"""metal""","""Follow You""","""Bring Me The Horizon"""
0.646,"""metal""","""Through Glass""","""Stone Sour"""
0.638,"""metal""","""Drive""","""Incubus"""


## Tuning note — HNSW recall/latency, and quantization

HNSW search is **approximate**, and Qdrant exposes the trade-off as query-time and build-time
knobs — worth knowing even though this notebook only *reads*:

- **`hnsw_ef`** (query-time) — how wide to search the graph. Higher `ef` gets closer to exact
  recall at the cost of latency; lower `ef` is faster but can miss true neighbours. A per-query
  dial, no re-indexing.
- **`m` / `ef_construct`** (build-time, seen above) — graph connectivity and build effort; they
  set the index's baseline recall and its memory footprint.
- **`exact=True`** — bypass the graph for a brute-force scan = ground truth, useful for
  *measuring* how much recall an ANN setting gives up.
- **Quantization** — Qdrant can store vectors as scalar/product/binary-compressed codes
  (`quantization_config`), trading a little recall for large memory savings and faster scans.
  These lab collections run **un-quantized** (`quantization: None`) — at thousands-to-100k
  points, in-memory float32 is simplest and the recall is free.

Below: the same search at a low `ef`, a high `ef`, and exact — plus the top-5 overlap between
the high-`ef` ANN and the exact ground truth (at this scale the graph is already essentially
exact).

In [9]:
def ids_of(points):
    return [str(p.id)[:8] for p in points]

ef_low  = client.query_points("weyland_chunks", query=seed_vec, limit=5,
                              search_params=models.SearchParams(hnsw_ef=8)).points
ef_high = client.query_points("weyland_chunks", query=seed_vec, limit=5,
                              search_params=models.SearchParams(hnsw_ef=256)).points
ground  = client.query_points("weyland_chunks", query=seed_vec, limit=5,
                              search_params=models.SearchParams(exact=True)).points

print("ANN hnsw_ef=8   :", ids_of(ef_low))
print("ANN hnsw_ef=256 :", ids_of(ef_high))
print("exact (brute)   :", ids_of(ground))

recall = len(set(ids_of(ef_high)) & set(ids_of(ground))) / len(ground)
print(f"\nhnsw_ef=256 vs exact top-5 overlap: {recall:.0%}")

ANN hnsw_ef=8   : ['0002ea0b', '5a4ad0cb', '6667615a', '0abe3317', '04fc69e7']
ANN hnsw_ef=256 : ['0002ea0b', '5a4ad0cb', '6667615a', '0abe3317', '04fc69e7']
exact (brute)   : ['0002ea0b', '5a4ad0cb', '6667615a', '0abe3317', '04fc69e7']

hnsw_ef=256 vs exact top-5 overlap: 100%


## When to reach for Qdrant

**Reach for Qdrant when the question is nearest-neighbour over embeddings, at scale, with
metadata constraints:**
- **RAG retrieval** — embed a query, fetch the nearest chunks, optionally scoped by a payload
  filter (source, tenant, recency). The centrepiece above, and Qdrant's core job in this mesh.
- **Semantic / similarity search** — "find things like this one" over text, audio-feature, or
  any embedding vectors, where keyword SQL cannot express *meaning*.
- **Filtered ANN at scale** — when the filter must apply *inside* the index, and when the
  collection is large enough that a filterable, quantizable, on-disk-capable HNSW earns its keep
  over vectors-in-a-column.

**Reach for pgvector instead when the vectors are a small side-feature of relational rows** you
already keep in Postgres and want to query transactionally in the same `SELECT` — no separate
store to run, at the cost of Qdrant's scale and index controls.

**Reach for the SQL engines (`20` Trino · `21` DuckDB · `22` native) when the question is over
rows and columns** — joins, aggregations, time-buckets. They answer "which rows match?"; Qdrant
answers "which vectors are nearest?" A full pipeline often uses both: SQL to assemble and filter
the corpus, Qdrant to retrieve by meaning.

| you want to… | use |
|--------------|-----|
| nearest-neighbour / semantic search over embeddings, filtered, at scale | **Qdrant** (this notebook) |
| a few vectors alongside relational rows, queried in SQL / transactionally | **pgvector** (in the operational Postgres) |
| joins / aggregations / time-series over rows and columns | **Trino · DuckDB · native clients** (`20` / `21` / `22`) |

Qdrant is the mesh's answer to **"retrieve by meaning, not by match"** — the vector index that
turns an embedding into its nearest real neighbours, with the metadata filters that make that
retrieval precise.